In [0]:
df = spark.read.csv('/Volumes/dev/dev_schema/myvolume/sales.csv', header=True , inferSchema=True)
df.display()
df.printSchema()
df.count()

In [0]:
# Find the total number of rows
# Find the total number of columns
# Find distinct customers
# Find distinct products
# Generate summary statistics

df.count()
df.columns


df.select('customer_id').distinct().display()
df.select('product_id').distinct().display()


df.summary().display()
df.describe().display()












In [0]:
# Step 3: Clean the Data
# Remove duplicate rows
# Remove duplicate records based on transaction_id
# Remove rows containing null values

# df.display()

df =df.dropDuplicates()
df = df.dropDuplicates(['transaction_id'])
df.display()
df = df.dropna()
df.display()

In [0]:

from pyspark.sql.functions import *
# df.display()


# # Rename columns
col_names = [
    "order_id",
    "customer_id",
    "transaction_id",
    "product_id",
    "quantity",
    "discount_amount",
    "total_amount",
    "order_date"
]

df = df.toDF(*col_names)

# # Derived column
df = df.withColumn(
    "net_amount",
    col("total_amount") - col("discount_amount")
)


df = df.withColumns({
    "current_timespamp": current_timestamp(),
    "current_date": current_date(),
    "file_name": col("_metadata.file_name"),
    "file_path": col("_metadata.file_path")
})
# # Metadata columns

df = df.withColumn(
    "order_date",
    to_date("order_date", "yyyy-MM-dd")
)
df.display()


In [0]:
# Step 5: Write the Data
# Write the cleaned data as Delta
# Create a Delta table from the cleaned data

df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dev.dev_schema.delta")
df = spark.read.table('dev.dev_schema.Delta')
df.display()



In [0]:
# Step 6: Analytics (SQL, PySpark)
#  Using the cleaned Delta table, answer the following business questions:
# Calculate the total sales.
# Find the total number of orders.
# Find the total number of unique customers.
# Calculate total sales for each customer.
# Calculate total sales for each product.
# Display the Top 10 customers by sales.
# Display the Top 10 products by sales.
# Find the highest sales transaction.
# Find the lowest sales transaction.
# Calculate the average order value.

# Store all the results in views 





# 1. Total Sales
df.select(
    sum("total_amount").alias("total_sales")
).createOrReplaceTempView("total_sales_view")


# 2. Total Number of Orders
df.select(
    countDistinct("order_id").alias("total_orders")
).createOrReplaceTempView("total_orders_view")


# 3. Total Unique Customers
df.select(
    countDistinct("customer_id").alias("unique_customers")
).createOrReplaceTempView("unique_customers_view")


# 4. Total Sales for Each Customer
df.groupBy("customer_id") \
  .agg(sum("total_amount").alias("total_sales")) \
  .createOrReplaceTempView("customer_sales_view")


# 5. Total Sales for Each Product
df.groupBy("product_id") \
  .agg(sum("total_amount").alias("total_sales")) \
  .createOrReplaceTempView("product_sales_view")


# 6. Top 10 Customers by Sales
df.groupBy("customer_id") \
  .agg(sum("total_amount").alias("total_sales")) \
  .orderBy(desc("total_sales")) \
  .limit(10) \
  .createOrReplaceTempView("top_10_customers_view")


# 7. Top 10 Products by Sales
df.groupBy("product_id") \
  .agg(sum("total_amount").alias("total_sales")) \
  .orderBy(desc("total_sales")) \
  .limit(10) \
  .createOrReplaceTempView("top_10_products_view")


# 8. Highest Sales Transaction
df.select(
    "transaction_id",
    "total_amount"
).orderBy(
    desc("total_amount")
).limit(1) \
 .createOrReplaceTempView("highest_sales_transaction_view")


# 9. Lowest Sales Transaction
df.select(
    "transaction_id",
    "total_amount"
).orderBy(
    asc("total_amount")
).limit(1) \
 .createOrReplaceTempView("lowest_sales_transaction_view")


# 10. Average Order Value
df.select(
    (
        sum("total_amount") / countDistinct("order_id")
    ).alias("average_order_value")
).createOrReplaceTempView("average_order_value_view")





